In [0]:
# ================================================================
# NOTEBOOK: nb_gold_daily_sales
# PURPOSE:  Daily category- and seller-level sales summary
#
# GRAIN:
#   One row per SaleDate × Category × SellerID
# ================================================================



from pyspark.sql import functions as F
from pyspark.sql.functions import col, when,round, count


#STORAGE as the base container path.
STORAGE = "abfss://source@stshopsensedevhj.dfs.core.windows.net"


# ================================================================
# STEP 1: READ ACTIVE SILVER DATA
# ================================================================


orders = (spark.read.format("delta").load(f"{STORAGE}/silver/orders/").filter(col("_is_deleted") == False))
items = (spark.read.format("delta").load(f"{STORAGE}/silver/orderitems/").filter(col("_is_deleted") == False))
sellers = (spark.read.format("delta").load(f"{STORAGE}/silver/sellers/").select("SellerID","SellerName","City","SellerTier","SellerSize"))


print(
    f"[READ] Orders: {orders.count()} | "
    f"Items: {items.count()} | "
    f"Sellers: {sellers.count()}"
)

# ================================================================
# STEP 2:
# Group order items by OrderID, Category and SellerID,
# then calculate revenue, discount, units, line items and average discount
# ================================================================


order_item_agg = (
    items.groupBy(
        "OrderID",
        "Category",
        "SellerID",
    )
    .agg(
        F.sum("TotalPrice").alias("CategoryNetRevenue"),
        F.sum("DiscountAmount").alias("CategoryDiscount"),
        F.sum("Quantity").alias("CategoryUnits"),
        F.count("OrderItemID").alias("CategoryLineItems"),
        F.avg("DiscountPct").alias("AvgDiscountPct")
    )
    .withColumn("CategoryGrossRevenue", col("CategoryNetRevenue") + col("CategoryDiscount"))
)


# ================================================================
# STEP 3: JOIN ORDERS, ITEMS AND SELLERS
# ================================================================

orders_for_gold = (
    orders.filter(col("IsDelivered") == True)
    .select("OrderID","CustomerID","OrderDate","PaymentMethod","IsPrimeOrder")
)

joined = (
    order_item_agg
    .join(orders_for_gold, on="OrderID", how="inner")   # keep only delivered orders that have matching item aggregates
    .join(sellers, on="SellerID", how="left")             # attach seller info where available
    .withColumn("SaleDate", F.to_date(col("OrderDate")))
)


# ================================================================
# STEP 4: DAILY AGGREGATION
# ================================================================


daily_sales = (
    joined
    .groupBy(
        "SaleDate",
        "Category",
        "SellerID",
        "SellerName",
        "SellerTier"
        ).agg(
            F.countDistinct("OrderID").alias("TotalOrders"),
            F.sum("CategoryGrossRevenue").alias("GrossRevenue"),
            F.sum("CategoryDiscount").alias("TotalDiscounts"),
            F.sum("CategoryNetRevenue").alias("NetRevenue"),
            F.sum("CategoryUnits").alias("UnitsSold"),
            F.sum("CategoryLineItems").alias("LineItems"),
            F.countDistinct("CustomerID").alias("UniqueCustomers"),
            F.countDistinct(
                F.when(col("IsPrimeOrder") == True,
                       col("OrderID")
                       )
                ).alias("PrimeOrders"),
            F.countDistinct(
                F.when(col("PaymentMethod") == "UPI",
                       col("OrderID")
                       )
                ).alias("UPIOrders"),
            F.countDistinct(F.when(
                col("PaymentMethod") == "CREDITCARD",
                col("OrderID")
                )
            ).alias("CardOrders"),
            F.countDistinct(
                F.when(
                    col("PaymentMethod") == "COD",
                    col("OrderID")
                    )
                ).alias("CODOrders")
            
        )
)
            
# ================================================================
# STEP 5: DERIVED COLUMNS
# ================================================================

daily_sales = (
    daily_sales
    .withColumn(
        "AvgOrderValue",
        F.when(
            col("TotalOrders") > 0,
            F.round(
                col("NetRevenue") / col("TotalOrders"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "AvgDiscountPct",
        F.when(
            col("GrossRevenue") > 0,
            F.round(
                col("TotalDiscounts")
                / col("GrossRevenue")
                * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
   .withColumn(
        "DiscountRate",
        F.when(
            col("GrossRevenue") > 0,
            F.round(col("TotalDiscounts") / col("GrossRevenue"), 4)
        ).otherwise(0.0)
    )
    .withColumn("PrimeOrderPct",
        F.when(col("TotalOrders") > 0,
               F.round(col("PrimeOrders") / col("TotalOrders") * 100,2)
        ).
        otherwise(F.lit(0.0))
    )
    .withColumn("Orderyear", F.year("SaleDate"))
    .withColumn("OrderMonth", F.month("SaleDate"))
    .withColumn("OrderDayOfWeek", F.dayofweek("SaleDate"))
    .withColumn("IsWeekendOrder", col("OrderDayOfWeek").isin([1, 7]))
    .withColumn("OrderDayOfWeek", F.dayofweek("SaleDate"))
    .withColumn("Quarter", F.quarter("SaleDate"))
    .withColumn("WeekOfYear", F.weekofyear("SaleDate"))
    .withColumn("_gold_load_ts", F.current_timestamp())
)

# ================================================================
# STEP 6: WRITE GOLD
# ================================================================            

(
    daily_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("OrderYear", "OrderMonth")
    .save(f"{STORAGE}/gold/daily_sales/")
)

# ================================================================
# STEP 7: VALIDATION
# ================================================================

total_rows = daily_sales.count()

total_revenue = (
    daily_sales
    .agg(F.sum("NetRevenue").alias("TotalRevenue"))
    .first()["TotalRevenue"]
)

minimum_date = (
    daily_sales
    .agg(F.min("SaleDate").alias("MinDate"))
    .first()["MinDate"]
)

maximum_date = (
    daily_sales
    .agg(F.max("SaleDate").alias("MaxDate"))
    .first()["MaxDate"]
)

print(f"\n[DONE] Gold daily_sales written: {total_rows} rows")

print(f"Total Net Revenue: ₹{total_revenue:,.2f}")


print(f"Date range: {minimum_date} to {maximum_date}")

print("\n[TOP CATEGORIES BY REVENUE]")

(
    daily_sales
    .groupBy("Category")
    .agg(
        F.round(
            F.sum("NetRevenue"),2).alias("Revenue")
    )
    .orderBy(
        col("Revenue").desc()
    )
    .show()
)


display(
    daily_sales
    .orderBy(
        col("SaleDate").desc(),
        col("NetRevenue").desc()
    )
    .limit(10)
)





[READ] Orders: 3015 | Items: 5945 | Sellers: 50

[DONE] Gold daily_sales written: 2452 rows
Total Net Revenue: ₹49,030,143.73
Date range: 2024-01-01 to 2024-06-29

[TOP CATEGORIES BY REVENUE]
+-----------+-----------+
|   Category|    Revenue|
+-----------+-----------+
|      BOOKS|10593544.41|
|   CLOTHING| 9937762.99|
|ELECTRONICS| 9770188.88|
|     BEAUTY| 9522353.70|
|HOMEKITCHEN| 9206293.75|
+-----------+-----------+



SaleDate,Category,SellerID,SellerName,SellerTier,TotalOrders,GrossRevenue,TotalDiscounts,NetRevenue,UnitsSold,LineItems,UniqueCustomers,PrimeOrders,UPIOrders,CardOrders,CODOrders,AvgOrderValue,AvgDiscountPct,DiscountRate,PrimeOrderPct,Orderyear,OrderMonth,OrderDayOfWeek,IsWeekendOrder,Quarter,WeekOfYear,_gold_load_ts
2024-06-29,HOMEKITCHEN,SELL018,Sportzone 18,PLATINUM,1,44458.88,0.00,44458.88,4,1,1,0,0,0,0,44458.88,0.0,0.0,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,CLOTHING,SELL013,Freshfoods 13,PLATINUM,1,39282.10,1402.93,37879.17,3,1,1,0,1,0,0,37879.17,3.57,0.0357,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,CLOTHING,SELL018,Sportzone 18,PLATINUM,1,34476.53,1231.31,33245.22,3,1,1,0,0,0,0,33245.22,3.57,0.0357,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,CLOTHING,SELL005,Freshfoods 5,PLATINUM,1,29720.77,1651.15,28069.62,3,1,1,0,1,0,0,28069.62,5.56,0.0556,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,CLOTHING,SELL034,Sportzone 34,PLATINUM,1,26618.70,345.70,26273.00,4,1,1,0,1,0,0,26273.0,1.3,0.013,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,ELECTRONICS,SELL008,Sportzone 8,PLATINUM,1,21470.72,1645.05,19825.67,7,2,1,0,1,0,0,19825.67,7.66,0.0766,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,ELECTRONICS,SELL013,Freshfoods 13,PLATINUM,1,20263.29,1642.97,18620.32,2,1,1,0,1,0,0,18620.32,8.11,0.0811,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,CLOTHING,SELL042,Homeessentials 42,PLATINUM,1,11397.51,0.00,11397.51,3,1,1,1,0,1,0,11397.51,0.0,0.0,100.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,HOMEKITCHEN,SELL013,Freshfoods 13,PLATINUM,1,4289.36,428.94,3860.42,1,1,1,0,1,0,0,3860.42,10.0,0.1,0.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
2024-06-29,BOOKS,SELL042,Homeessentials 42,PLATINUM,1,3587.24,179.36,3407.88,1,1,1,1,0,1,0,3407.88,5.0,0.05,100.0,2024,6,7,true,2,26,2026-07-16T19:22:43.324Z
